# Step 12: Sparse Autoencoders — Recovering Features from Superposition

Notebook 11 showed that neural networks store many features in few neurons via
superposition. Individual neurons are **polysemantic** — they respond to multiple
unrelated features.

This makes mechanistic interpretability hard: you can't understand what a model is doing
by looking at neurons, because features ≠ neurons.

**Sparse Autoencoders (SAEs)** are a post-hoc technique to *undo* superposition:
train an overcomplete autoencoder with a sparsity penalty on the hidden layer.
The hidden units learn to correspond to the underlying monosemantic features.

This notebook reproduces the core approach from:
> Bricken et al. (2023). *Towards Monosemanticity: Decomposing Language Models
> With Dictionary Learning.* transformer-circuits.pub

We:
1. Create a polysemantic toy model with known ground-truth features
2. Train an SAE on its activations
3. Show the SAE recovers the original features
4. Apply the same technique to a small trained transformer

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

torch.manual_seed(42)
np.random.seed(42)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

device = torch.device('cpu')

## The SAE Architecture

A sparse autoencoder maps activations to a larger feature space and back:

```
x (n_neurons)  →  f = ReLU(Wₑ(x - b_d) + bₑ)  →  n_features hidden units
                  x̂ = b_d + Σᵢ f(x)ᵢ dᵢ        →  n_neurons (reconstruction)
```

Key design choices from Bricken et al.:
- **Overcomplete**: n_features >> n_neurons (expansion factor 4×–256×)
- **Pre-encoder bias**: subtract b_d before encoding — removes the "average activation" direction
- **Decoder columns normalized to unit length**: each column dᵢ is a feature *direction*
- **L1 penalty**: forces sparse activation — only a few features active per input
- **Loss**: L = ||x - x̂||² + λ · ||f||₁

The L1 term is the key: it forces the SAE to explain each activation as a sparse combination
of feature directions. If superposition packs many features into few neurons, the SAE
learns to *unpack* them.

In [ ]:
class SparseAutoencoder(nn.Module):
    """
    Sparse autoencoder following Bricken et al. 2023.

    n_input:    dimensionality of activations to decompose (= n_neurons in the base model)
    n_features: dictionary size (should be > n_input, often much larger)
    """
    def __init__(self, n_input, n_features):
        super().__init__()
        # Pre-encoder bias: learned estimate of mean activation
        self.b_d = nn.Parameter(torch.zeros(n_input))
        # Encoder weights: (n_features, n_input)
        self.W_e = nn.Parameter(torch.randn(n_features, n_input) * 0.1)
        # Encoder bias
        self.b_e = nn.Parameter(torch.zeros(n_features))
        # Decoder columns: (n_input, n_features) — will be normalized after each step
        self.W_d = nn.Parameter(torch.randn(n_input, n_features) * 0.1)

        self.n_input = n_input
        self.n_features = n_features

    def encode(self, x):
        """f(x) = ReLU(Wₑ(x - b_d) + bₑ)"""
        return F.relu((x - self.b_d) @ self.W_e.T + self.b_e)

    def decode(self, f):
        """x̂ = b_d + f @ W_d^T"""
        # Normalize decoder columns to unit length before decoding
        W_d_norm = self.W_d / (self.W_d.norm(dim=0, keepdim=True) + 1e-8)
        return self.b_d + f @ W_d_norm.T

    def forward(self, x):
        f = self.encode(x)
        x_hat = self.decode(f)
        return x_hat, f

    @torch.no_grad()
    def normalize_decoder(self):
        """Normalize decoder columns to unit length. Call after each optimizer step."""
        self.W_d.data = self.W_d.data / (self.W_d.data.norm(dim=0, keepdim=True) + 1e-8)


def train_sae(sae, activations_tensor, l1_coeff=1e-3, n_steps=2000,
              batch_size=256, lr=1e-3, verbose=True):
    """
    Train an SAE on a tensor of activations.
    activations_tensor: (n_samples, n_input)
    Returns: list of (step, loss, l2_loss, l1_loss, n_dead_features)
    """
    optimizer = torch.optim.Adam(sae.parameters(), lr=lr)
    n_samples = activations_tensor.shape[0]
    log = []

    for step in range(n_steps):
        # Random batch
        idx = torch.randint(0, n_samples, (batch_size,))
        x = activations_tensor[idx]

        x_hat, f = sae(x)
        l2_loss = ((x - x_hat) ** 2).sum(dim=-1).mean()
        l1_loss = f.abs().sum(dim=-1).mean()
        loss = l2_loss + l1_coeff * l1_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        sae.normalize_decoder()

        if step % 500 == 0:
            with torch.no_grad():
                f_all = sae.encode(activations_tensor[:1000])
                n_dead = (f_all.max(dim=0).values < 1e-4).sum().item()
            log.append((step, loss.item(), l2_loss.item(), l1_loss.item(), n_dead))
            if verbose:
                print(f"Step {step:4d} | loss={loss.item():.4f} "
                      f"l2={l2_loss.item():.4f} l1={l1_loss.item():.4f} "
                      f"dead={n_dead}/{sae.n_features}")

    return log


print("SAE class defined.")
# Quick sanity check
sae_test = SparseAutoencoder(n_input=8, n_features=32)
x_test = torch.randn(10, 8)
x_hat, f = sae_test(x_test)
print(f"Input shape: {x_test.shape} → features: {f.shape} → reconstruction: {x_hat.shape}")
print(f"Expansion factor: {32/8}×")

## Part 1: Recovering Known Features from a Polysemantic Model

First, let's create a situation where we *know* the ground truth:

1. Define **n_true = 12 ground-truth features** (sparse, independent)
2. Project them into **n_neurons = 4 dimensions** via a random matrix (this is the "model")
3. Train an SAE on the 4-dimensional activations
4. Check: do the SAE features match the original 12 feature directions?

This is a ground-truth recovery experiment — we can measure success precisely.

In [ ]:
# Ground-truth setup
n_true = 12       # True features in the world
n_neurons = 4     # Bottleneck ("neurons" of the model we're interpreting)
n_sae = 32        # SAE dictionary size (8× expansion over neurons)
sparsity = 0.85   # Each feature inactive 85% of the time
n_samples = 50000

# Ground-truth feature directions: (n_neurons, n_true)
# Random unit vectors — this is the "true" feature basis the model uses
torch.manual_seed(1)
W_true = torch.randn(n_neurons, n_true)
W_true = W_true / W_true.norm(dim=0, keepdim=True)  # normalize columns

# Generate sparse feature activations
mask = torch.bernoulli(torch.full((n_samples, n_true), 1 - sparsity))
values = torch.rand(n_samples, n_true)
features_true = mask * values   # (n_samples, n_true)

# "Model activations": project features into neuron space
# activations = features_true @ W_true^T  — shape (n_samples, n_neurons)
activations = features_true @ W_true.T
# Add small noise to simulate real model imperfections
activations = activations + 0.02 * torch.randn_like(activations)

print(f"Ground truth: {n_true} features → {n_neurons} neurons")
print(f"Mean active features per sample: {(mask.sum(dim=1) > 0).float().mean() * n_true:.1f}")
print(f"Activations shape: {activations.shape}")
print(f"\nNow training SAE with {n_sae} features ({n_sae/n_neurons}× expansion)...")

sae = SparseAutoencoder(n_input=n_neurons, n_features=n_sae)
log = train_sae(sae, activations, l1_coeff=5e-3, n_steps=3000, batch_size=512)

In [ ]:
# Evaluate: how well did the SAE recover the ground-truth features?

# Get learned feature directions (decoder columns, already unit-normalized)
with torch.no_grad():
    W_d = sae.W_d.data  # (n_neurons, n_sae)
    W_d_norm = W_d / (W_d.norm(dim=0, keepdim=True) + 1e-8)

# For each ground-truth feature, find the SAE feature most similar to it
# W_true: (n_neurons, n_true)  —  W_d_norm: (n_neurons, n_sae)
cosine_sim_matrix = W_true.T @ W_d_norm  # (n_true, n_sae)
best_match_sim = cosine_sim_matrix.abs().max(dim=1).values  # (n_true,)

print("Recovery of ground-truth features:")
print(f"Mean max cosine similarity: {best_match_sim.mean():.3f} (1.0 = perfect recovery)")
print(f"Features recovered (sim > 0.8): {(best_match_sim > 0.8).sum().item()} / {n_true}")
print(f"Features recovered (sim > 0.9): {(best_match_sim > 0.9).sum().item()} / {n_true}")

# Also check dead features
with torch.no_grad():
    f_all = sae.encode(activations[:5000])
    n_dead = (f_all.max(dim=0).values < 1e-4).sum().item()
    mean_active = (f_all > 0).float().sum(dim=1).mean().item()
print(f"Dead SAE features: {n_dead} / {n_sae}")
print(f"Mean active features per sample: {mean_active:.1f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: cosine similarity of each ground-truth feature to its best SAE match
ax = axes[0]
colors = ['steelblue' if s > 0.8 else 'salmon' for s in best_match_sim.numpy()]
ax.bar(range(n_true), best_match_sim.numpy(), color=colors)
ax.axhline(0.8, color='gray', linestyle='--', alpha=0.7, label='Recovery threshold (0.8)')
ax.set_xlabel('Ground-truth feature index')
ax.set_ylabel('Max cosine similarity with any SAE feature')
ax.set_title('SAE Feature Recovery Quality', fontsize=11)
ax.legend()
ax.set_ylim(0, 1.05)

# Right: heatmap of cosine similarities (ground-truth vs. SAE features)
# Sort SAE features to show only the best matches
ax = axes[1]
best_sae_idx = cosine_sim_matrix.abs().argmax(dim=1)  # (n_true,)
# Select the SAE features that are best matches to any ground-truth feature
unique_matches = torch.unique(best_sae_idx)
sub_matrix = cosine_sim_matrix[:, unique_matches].numpy()  # (n_true, n_matched)

im = ax.imshow(np.abs(sub_matrix), cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
ax.set_xlabel(f'Matched SAE features (top {len(unique_matches)} of {n_sae})')
ax.set_ylabel('Ground-truth feature')
ax.set_yticks(range(n_true))
ax.set_yticklabels([f'GT-{i}' for i in range(n_true)])
plt.colorbar(im, ax=ax, label='|Cosine similarity|')
ax.set_title('Ground Truth ↔ SAE Feature Alignment', fontsize=11)

plt.suptitle(f'SAE Recovery of {n_true} Features from {n_neurons}-Neuron Activations', fontsize=12)
plt.tight_layout()
plt.savefig('12_feature_recovery.png', dpi=100, bbox_inches='tight')
plt.show()

## Part 2: The L1 Coefficient Trade-Off

The L1 penalty strength λ is the key hyperparameter:

- **Too small λ**: SAE uses all features weakly (like PCA). Reconstruction is great but
  features are dense and hard to interpret.
- **Too large λ**: SAE uses very few features per sample. Reconstruction degrades.
  Dead features accumulate.
- **Just right**: Each sample uses ~3–5 features. Each feature fires on a specific
  subset of inputs. Reconstruction is good. Features are monosemantic.

Let's sweep λ and measure the trade-off.

In [ ]:
l1_coeffs = [0.0, 1e-4, 5e-4, 1e-3, 5e-3, 1e-2, 5e-2]
results = []

for l1 in l1_coeffs:
    torch.manual_seed(42)
    sae_sweep = SparseAutoencoder(n_input=n_neurons, n_features=n_sae)
    train_sae(sae_sweep, activations, l1_coeff=l1, n_steps=2000,
              batch_size=512, verbose=False)

    with torch.no_grad():
        f_all = sae_sweep.encode(activations[:5000])
        x_hat_all, _ = sae_sweep(activations[:5000])

        # Reconstruction quality
        recon_mse = ((activations[:5000] - x_hat_all) ** 2).mean().item()

        # Sparsity: mean number of active features per sample
        mean_active = (f_all > 0).float().sum(dim=1).mean().item()

        # Dead features
        n_dead = (f_all.max(dim=0).values < 1e-4).sum().item()

        # Feature recovery
        W_d = sae_sweep.W_d.data
        W_d_norm = W_d / (W_d.norm(dim=0, keepdim=True) + 1e-8)
        csim = W_true.T @ W_d_norm
        recovery = (csim.abs().max(dim=1).values > 0.8).float().mean().item()

    results.append(dict(l1=l1, mse=recon_mse, mean_active=mean_active,
                        n_dead=n_dead, recovery=recovery))
    print(f"λ={l1:.0e} | mse={recon_mse:.4f} active={mean_active:.1f} "
          f"dead={n_dead} recovery={recovery:.2f}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

l1_vals = [r['l1'] for r in results]
x_pos = range(len(l1_vals))
l1_labels = [f'{v:.0e}' if v > 0 else '0' for v in l1_vals]

# Reconstruction MSE
ax = axes[0, 0]
ax.plot(x_pos, [r['mse'] for r in results], 'o-', color='steelblue', linewidth=2)
ax.set_xticks(x_pos); ax.set_xticklabels(l1_labels, rotation=45)
ax.set_xlabel('L1 coefficient λ'); ax.set_ylabel('Reconstruction MSE')
ax.set_title('Reconstruction Quality (lower = better)', fontsize=11)

# Mean active features
ax = axes[0, 1]
ax.plot(x_pos, [r['mean_active'] for r in results], 'o-', color='orange', linewidth=2)
ax.axhline(n_true * (1 - sparsity), color='gray', linestyle='--', alpha=0.6,
           label=f'True mean active ({n_true*(1-sparsity):.1f})')
ax.set_xticks(x_pos); ax.set_xticklabels(l1_labels, rotation=45)
ax.set_xlabel('L1 coefficient λ'); ax.set_ylabel('Mean active SAE features / sample')
ax.set_title('Feature Sparsity (lower = sparser)', fontsize=11)
ax.legend(fontsize=9)

# Dead features
ax = axes[1, 0]
ax.bar(x_pos, [r['n_dead'] for r in results], color='salmon')
ax.set_xticks(x_pos); ax.set_xticklabels(l1_labels, rotation=45)
ax.set_xlabel('L1 coefficient λ'); ax.set_ylabel('Dead features (never active)')
ax.set_title(f'Dead Features / {n_sae} Total', fontsize=11)

# Feature recovery
ax = axes[1, 1]
ax.plot(x_pos, [r['recovery'] for r in results], 'o-', color='green', linewidth=2)
ax.axhline(1.0, color='gray', linestyle='--', alpha=0.4, label='Perfect recovery')
ax.set_xticks(x_pos); ax.set_xticklabels(l1_labels, rotation=45)
ax.set_xlabel('L1 coefficient λ')
ax.set_ylabel(f'Fraction of {n_true} GT features recovered (cosine > 0.8)')
ax.set_title('Ground-Truth Feature Recovery', fontsize=11)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=9)

plt.suptitle('L1 Coefficient Trade-off: Reconstruction vs. Sparsity vs. Recovery', fontsize=12)
plt.tight_layout()
plt.savefig('12_l1_tradeoff.png', dpi=100, bbox_inches='tight')
plt.show()
print("The sweet spot: enough L1 to force sparsity, but not so much that features die.")

## Part 3: SAE on a Trained Transformer MLP

Now let's apply the SAE to something closer to the real use case: a tiny transformer
trained on a character-level language modeling task.

We'll:
1. Train a tiny transformer (2-layer, d_model=32) on a synthetic sequence task
2. Collect MLP activations from the first layer
3. Train an SAE on those activations
4. Analyze what features the SAE finds

The synthetic task: predict the next character in sequences containing structured
sub-patterns (numbers, letters, and mixed). The model should learn features
corresponding to these patterns.

In [ ]:
# --- Synthetic dataset ---
# Characters: digits 0-9, letters a-z, and some punctuation
# Sequences consist of: [all digits], [all letters], [alternating], [random]
# The model should learn context features for each regime.

VOCAB = list('0123456789abcdefghijklmnopqrstuvwxyz .,!?')
VOCAB_SIZE = len(VOCAB)
ch2idx = {c: i for i, c in enumerate(VOCAB)}
idx2ch = {i: c for c, i in ch2idx.items()}

DIGITS = list('0123456789')
LETTERS = list('abcdefghijklmnopqrstuvwxyz')

def make_sequence(seq_type, length=16):
    """Generate a sequence with a known structure type."""
    if seq_type == 'digits':
        return [np.random.choice(DIGITS) for _ in range(length)]
    elif seq_type == 'letters':
        return [np.random.choice(LETTERS) for _ in range(length)]
    elif seq_type == 'alternating':
        result = []
        for i in range(length):
            result.append(np.random.choice(DIGITS if i % 2 == 0 else LETTERS))
        return result
    else:  # random
        return [np.random.choice(VOCAB[:36]) for _ in range(length)]


SEQ_LEN = 16
SEQ_TYPES = ['digits', 'letters', 'alternating', 'random']

def make_dataset(n_seqs_per_type=2000):
    seqs, labels_list = [], []
    for t in SEQ_TYPES:
        for _ in range(n_seqs_per_type):
            s = make_sequence(t, SEQ_LEN)
            seqs.append([ch2idx[c] for c in s])
            labels_list.append(SEQ_TYPES.index(t))
    seqs_t = torch.tensor(seqs, dtype=torch.long)
    labels_t = torch.tensor(labels_list, dtype=torch.long)
    return seqs_t, labels_t

train_seqs, train_labels = make_dataset(2000)
print(f"Dataset: {train_seqs.shape} sequences, {VOCAB_SIZE} vocab")
print(f"Example digit sequence: {''.join(idx2ch[i.item()] for i in train_seqs[0])}")
print(f"Example letter sequence: {''.join(idx2ch[i.item()] for i in train_seqs[2001])}")
print(f"Example alternating:     {''.join(idx2ch[i.item()] for i in train_seqs[4001])}")

In [ ]:
class TinyTransformerWithHooks(nn.Module):
    """
    Small 1-layer transformer for character-level LM.
    Includes activation hooks so we can capture MLP hidden states.
    """
    def __init__(self, vocab_size, d_model=48, n_heads=4, d_ff=64, seq_len=SEQ_LEN):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(seq_len, d_model)
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp_in = nn.Linear(d_model, d_ff)
        self.mlp_out = nn.Linear(d_ff, d_model)
        self.ln3 = nn.LayerNorm(d_model)
        self.unembed = nn.Linear(d_model, vocab_size)

        self.d_ff = d_ff
        self._mlp_acts = None  # will be populated during forward pass

    def forward(self, x, return_mlp_acts=False):
        B, T = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0)
        h = self.embed(x) + self.pos_embed(pos)

        # Causal mask
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()

        # Attention block
        h2, _ = self.attn(self.ln1(h), self.ln1(h), self.ln1(h), attn_mask=mask)
        h = h + h2

        # MLP block — capture pre-ReLU and post-ReLU activations
        h_ln = self.ln2(h)
        mlp_pre = self.mlp_in(h_ln)  # (B, T, d_ff)
        mlp_acts = F.relu(mlp_pre)    # (B, T, d_ff) — what the SAE will decompose
        self._mlp_acts = mlp_acts
        h = h + self.mlp_out(mlp_acts)

        logits = self.unembed(self.ln3(h))  # (B, T, vocab_size)
        return logits


def train_transformer(model, seqs, n_steps=3000, batch_size=64, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    n = len(seqs)
    losses = []
    for step in range(n_steps):
        idx = torch.randint(0, n, (batch_size,))
        batch = seqs[idx]  # (B, SEQ_LEN)
        x = batch[:, :-1]
        targets = batch[:, 1:]
        logits = model(x)
        loss = F.cross_entropy(logits.reshape(-1, VOCAB_SIZE), targets.reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if step % 500 == 0:
            losses.append(loss.item())
            print(f"Step {step}: loss={loss.item():.3f}")
    return losses


torch.manual_seed(0)
transformer = TinyTransformerWithHooks(VOCAB_SIZE, d_model=48, n_heads=4, d_ff=64)
print(f"Transformer parameters: {sum(p.numel() for p in transformer.parameters()):,}")
print("\nTraining transformer...")
losses = train_transformer(transformer, train_seqs)

In [ ]:
# --- Collect MLP activations ---
print("Collecting MLP activations...")
transformer.eval()
all_acts = []
all_labels = []

with torch.no_grad():
    batch_size = 128
    for start in range(0, len(train_seqs), batch_size):
        batch = train_seqs[start:start+batch_size]
        lbls = train_labels[start:start+batch_size]
        x = batch[:, :-1]  # (B, SEQ_LEN-1)
        _ = transformer(x)
        acts = transformer._mlp_acts  # (B, SEQ_LEN-1, d_ff)
        # Flatten: treat each (sequence, position) pair as a separate sample
        B, T, D = acts.shape
        all_acts.append(acts.reshape(B * T, D))
        # Repeat label for each position
        all_labels.append(lbls.unsqueeze(1).expand(-1, T).reshape(-1))

all_acts = torch.cat(all_acts, dim=0)      # (N, d_ff)
all_labels = torch.cat(all_labels, dim=0)  # (N,)

print(f"Collected {all_acts.shape[0]:,} activation vectors of size {all_acts.shape[1]}")
print(f"Mean activation: {all_acts.mean():.3f}, Std: {all_acts.std():.3f}")
print(f"Fraction of activations > 0 (post-ReLU): {(all_acts > 0).float().mean():.3f}")

In [ ]:
# --- Train the SAE on transformer MLP activations ---
D_MLP = all_acts.shape[1]  # = d_ff = 64
N_SAE_FEATS = 256           # 4× expansion

print(f"Training SAE: {D_MLP} neurons → {N_SAE_FEATS} features ({N_SAE_FEATS/D_MLP}× expansion)")
torch.manual_seed(42)
sae_transformer = SparseAutoencoder(n_input=D_MLP, n_features=N_SAE_FEATS)
log_t = train_sae(sae_transformer, all_acts, l1_coeff=2e-3, n_steps=3000, batch_size=512)

In [ ]:
# --- Analyze learned SAE features ---
# For each feature: what sequence types activate it most?

sae_transformer.eval()
with torch.no_grad():
    f_all = sae_transformer.encode(all_acts)  # (N, N_SAE_FEATS)

# Feature activation rates by sequence type
type_names = SEQ_TYPES
n_types = len(type_names)
feat_by_type = torch.zeros(n_types, N_SAE_FEATS)

for t, tname in enumerate(type_names):
    type_mask = (all_labels == t)
    if type_mask.sum() > 0:
        # Mean activation of each SAE feature on this sequence type
        feat_by_type[t] = f_all[type_mask].mean(dim=0)

# Find "selective" features: features that activate much more for one type than others
# Selectivity = max_type_activation / mean_across_types
max_act = feat_by_type.max(dim=0).values
mean_act = feat_by_type.mean(dim=0)
selectivity = max_act / (mean_act + 1e-8)

# Dead features: never activate
n_dead_t = (f_all.max(dim=0).values < 1e-4).sum().item()
print(f"Dead features: {n_dead_t} / {N_SAE_FEATS}")
print(f"Mean active features per sample: {(f_all > 0).float().sum(dim=1).mean():.1f}")

# Select top selective features
live_mask = f_all.max(dim=0).values > 1e-4
top_selective_idx = (selectivity * live_mask.float()).topk(16).indices

print(f"\nTop selective features (activating strongly for one sequence type):")
for i, feat_idx in enumerate(top_selective_idx[:8]):
    feat_idx = feat_idx.item()
    dominant_type = feat_by_type[:, feat_idx].argmax().item()
    acts = feat_by_type[:, feat_idx].numpy()
    print(f"  Feature {feat_idx:3d}: dominant={type_names[dominant_type]:12s} "
          f"acts={[f'{a:.3f}' for a in acts]}")

In [ ]:
# Visualize: for each sequence type, how does the SAE activate?
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Top: activation heatmap for most selective features by type
# Find top 8 most type-specific features for each type
n_show = 32  # Show these many features in the heatmap

# Get top features ranked by max activation across types
top_feat_by_max = max_act.topk(n_show).indices.sort().values
sub_matrix = feat_by_type[:, top_feat_by_max].numpy()  # (4, n_show)

ax = axes[0, 0]
im = ax.imshow(sub_matrix, cmap='YlOrRd', aspect='auto')
ax.set_yticks(range(n_types))
ax.set_yticklabels(type_names)
ax.set_xlabel('SAE feature (top 32 by max activation)')
ax.set_title('Mean SAE Feature Activation by Sequence Type', fontsize=10)
plt.colorbar(im, ax=ax, label='Mean activation')

# Second: selectivity histogram
ax = axes[0, 1]
live_selectivity = selectivity[live_mask].numpy()
ax.hist(live_selectivity, bins=30, color='steelblue', alpha=0.8)
ax.axvline(3.0, color='red', linestyle='--', alpha=0.7, label='Selectivity > 3 (type-specific)')
n_selective = (live_selectivity > 3).sum()
ax.set_xlabel('Selectivity (max_type / mean_type)')
ax.set_ylabel('Number of SAE features')
ax.set_title(f'Feature Selectivity Distribution\n({n_selective} features are type-selective)', fontsize=10)
ax.legend()

# Third: feature activation examples — show activations for 4 specific sequences
ax = axes[1, 0]
# Get one example sequence of each type
example_acts = []
example_labels = []
with torch.no_grad():
    for t, tname in enumerate(type_names):
        seq_chars = make_sequence(tname, SEQ_LEN)
        seq_idx = torch.tensor([[ch2idx[c] for c in seq_chars]])
        _ = transformer(seq_idx[:, :-1])
        acts_ex = transformer._mlp_acts  # (1, T-1, D)
        # Use middle position
        mid = (SEQ_LEN - 1) // 2
        example_acts.append(sae_transformer.encode(acts_ex[0, mid:mid+1]))
        example_labels.append(tname)

example_acts_t = torch.cat(example_acts, dim=0)  # (4, N_SAE_FEATS)

# Show only top-selective features
top_sel_feats = top_selective_idx[:24].sort().values
sub_ex = example_acts_t[:, top_sel_feats].numpy()

im2 = ax.imshow(sub_ex, cmap='Blues', aspect='auto')
ax.set_yticks(range(4))
ax.set_yticklabels(example_labels)
ax.set_xlabel('SAE feature (top 24 selective)')
ax.set_title('SAE Activation on Individual Example Sequences\n(middle position)', fontsize=10)
plt.colorbar(im2, ax=ax)

# Fourth: reconstruction quality on test set
ax = axes[1, 1]
with torch.no_grad():
    test_sample = all_acts[:1000]
    x_hat_test, f_test = sae_transformer(test_sample)
    per_sample_mse = ((test_sample - x_hat_test) ** 2).mean(dim=1).numpy()
    baseline_mse = ((test_sample - test_sample.mean(dim=0, keepdim=True)) ** 2).mean(dim=1).numpy()

ax.scatter(range(200), per_sample_mse[:200], alpha=0.5, s=8, label='SAE reconstruction', color='steelblue')
ax.axhline(np.mean(baseline_mse[:200]), color='red', linestyle='--', alpha=0.7,
           label=f'Baseline (mean) MSE = {np.mean(baseline_mse[:200]):.3f}')
ax.axhline(np.mean(per_sample_mse), color='green', linestyle='--', alpha=0.7,
           label=f'SAE mean MSE = {np.mean(per_sample_mse):.3f}')
ax.set_xlabel('Sample index'); ax.set_ylabel('MSE')
ax.set_title('Reconstruction Quality per Sample', fontsize=10)
ax.legend(fontsize=8)

plt.suptitle('SAE Applied to Tiny Transformer MLP Activations', fontsize=12)
plt.tight_layout()
plt.savefig('12_transformer_sae.png', dpi=100, bbox_inches='tight')
plt.show()

## Part 4: Feature Interpretability — What Does Each Feature Detect?

The real test: can we describe what an SAE feature does?

For each feature, we'll look at:
1. Which input positions activate it most strongly (top activating examples)
2. What output tokens it predicts (logit effects)
3. Whether it's monosemantic (specific) or polysemantic (vague)

In [ ]:
# For each of the top selective features, show what characters activate it most

# First, collect token-level info alongside activations
# Re-collect with token info
token_acts = []  # (N, D_MLP)
token_chars = [] # (N,) - which character was at this position
token_types = [] # (N,) - sequence type

transformer.eval()
with torch.no_grad():
    for start in range(0, min(len(train_seqs), 4000), 64):
        batch = train_seqs[start:start+64]
        lbls = train_labels[start:start+64]
        x = batch[:, :-1]
        _ = transformer(x)
        acts = transformer._mlp_acts  # (B, T-1, D)
        B, T, D = acts.shape
        token_acts.append(acts.reshape(B*T, D))
        # Which character is at each position (the input token)
        chars = x.reshape(B*T).numpy()
        token_chars.extend(chars)
        token_types.extend(lbls.unsqueeze(1).expand(-1, T).reshape(-1).numpy())

token_acts_t = torch.cat(token_acts, dim=0)
token_chars_arr = np.array(token_chars)
token_types_arr = np.array(token_types)

# Encode with SAE
with torch.no_grad():
    f_tokens = sae_transformer.encode(token_acts_t)  # (N, N_SAE_FEATS)

# Identify top selective features
feat_type_acts = torch.zeros(n_types, N_SAE_FEATS)
for t in range(n_types):
    mask = torch.tensor(token_types_arr == t)
    if mask.sum() > 0:
        feat_type_acts[t] = f_tokens[mask].mean(dim=0)

max_type_act = feat_type_acts.max(dim=0).values
mean_type_act = feat_type_acts.mean(dim=0)
selectivity2 = max_type_act / (mean_type_act + 1e-8)
live_mask2 = f_tokens.max(dim=0).values > 1e-4
top_feats = (selectivity2 * live_mask2.float()).topk(6).indices

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for ax, feat_idx in zip(axes.flatten(), top_feats):
    feat_idx = feat_idx.item()
    feat_acts = f_tokens[:, feat_idx].numpy()

    # Mean activation per character
    char_mean_act = np.zeros(VOCAB_SIZE)
    char_count = np.zeros(VOCAB_SIZE)
    for i, (c, a) in enumerate(zip(token_chars_arr, feat_acts)):
        char_mean_act[c] += a
        char_count[c] += 1
    with np.errstate(invalid='ignore'):
        char_mean_act = np.where(char_count > 0, char_mean_act / char_count, 0)

    # Plot: activation by character
    sorted_idx = np.argsort(char_mean_act)[::-1][:15]
    top_chars = [VOCAB[i] for i in sorted_idx]
    top_acts = [char_mean_act[i] for i in sorted_idx]

    bars = ax.bar(range(len(top_chars)), top_acts,
                  color=['steelblue' if c.isdigit() else
                         'green' if c.isalpha() else 'gray'
                         for c in top_chars])
    ax.set_xticks(range(len(top_chars)))
    ax.set_xticklabels([repr(c)[1:-1] for c in top_chars], fontsize=8)

    dominant = type_names[feat_type_acts[:, feat_idx].argmax().item()]
    ax.set_title(f'Feature {feat_idx} (dominant: {dominant})\nblue=digit, green=letter',
                 fontsize=9)
    ax.set_ylabel('Mean activation', fontsize=8)

plt.suptitle('Top SAE Features: What Characters Activate Each Feature?', fontsize=12)
plt.tight_layout()
plt.savefig('12_feature_characters.png', dpi=100, bbox_inches='tight')
plt.show()
print("Blue bars = digit characters, Green = letters")
print("A monosemantic feature should activate selectively for one character type.")

## Connection to Bricken et al. 2023

Our toy experiments mirror the structure of the real paper:

| This notebook | Bricken et al. |
|---|---|
| Synthetic 4-neuron model with 12 known features | 512-neuron MLP of a one-layer transformer |
| SAE with 32 features (8× expansion) | SAE with 4,096 features (8× expansion = A/1) |
| Ground-truth recovery metric | Arabic/DNA/base64 features verified by ablation |
| Character-type features (digit/letter) | Script-type features (Arabic, Hebrew, base64) |
| L1 sweep showing reconstruction/sparsity trade-off | Same trade-off studied empirically |

The key differences are scale and verification rigor:
- The paper uses 8 billion training samples (vs. our ~50,000)
- Features are verified *causally* — ablating them degrades model behavior as predicted
- The paper's automated interpretability pipeline uses Claude to generate
  human-readable descriptions and predict activations on held-out tokens

But the core mechanism is the same: L1-sparse overcomplete autoencoders can unfold
superposition, recovering interpretable monosemantic features from polysemantic neurons.

## Key Takeaways

| Concept | What we learned |
|---|---|
| **SAE architecture** | Overcomplete encoder (ReLU + L1) + unit-norm decoder; pre-encoder bias removes mean activation |
| **L1 coefficient** | Controls the reconstruction/sparsity trade-off; too high kills features, too low = dense PCA |
| **Feature recovery** | With right λ, SAE recovers ground-truth feature directions with >80% cosine similarity |
| **Dead features** | Some SAE features never activate; L1 too high → more dead features |
| **Monosemanticity** | Recovered features are type-specific (digit vs. letter) while original neurons are not |
| **Feature splitting** | More SAE features → finer-grained distinctions within a category |
| **Safety path** | Once features are interpretable and enumerable, you can monitor/audit model behavior |

**What this enables**:
- Identify which features activate on safety-relevant inputs (deception, harm, etc.)
- Ablate specific features to verify their causal role
- Search for dangerous circuits in the feature-space graph
- Build toward **enumerative safety**: prove no harmful circuit exists in the model

**Open challenges** (as of the Bricken et al. paper):
- Scaling SAEs to 100B+ parameter models (millions of features needed)
- Token-in-context features: are they real or SAE artifacts from over-sparsity?
- Universality: do the same features appear across different model scales?
- Full circuit analysis in feature space (not just individual features but how they connect)